# FT-Transformer & TabPFN Model Comparison
## 5-Fold Cross-Validation with Optuna Hyperparameter Tuning

이 노트북은 icu_final_feature_set.csv 데이터에 대해 FT-Transformer와 TabPFN 모델을 비교합니다.
- **Base 모델**: StandardScaler 적용 후 기본 설정으로 학습
- **Tuned 모델**: Optuna를 사용한 5-fold 교차검증으로 최적화

In [ ]:
# 필요한 패키지 설치
!pip install -q torch optuna scikit-learn pandas numpy tabpfn

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
from tabpfn import TabPFNRegressor
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')

In [ ]:
# 데이터 로드
df = pd.read_csv('icu_final_feature_set.csv')
print(f'데이터 형상: {df.shape}')
print(f'\n컬럼 정보:')
print(df.info())
print(f'\n처음 5개 행:')
print(df.head())

In [ ]:
# 데이터 전처리
df_proc = df.copy()

# 범주형 변수 인코딩
categorical_cols = ['gender', 'race_simplified']
for col in categorical_cols:
    if col in df_proc.columns:
        df_proc = pd.get_dummies(df_proc, columns=[col], drop_first=True)

# 타겟 변수 분리 (los 컬럼이 타겟이라고 가정)
if 'los' in df_proc.columns:
    y = df_proc['los'].values
    X = df_proc.drop(columns=['los']).values
    feature_names = df_proc.drop(columns=['los']).columns.tolist()
else:
    # los 컬럼이 없으면 마지막 컬럼을 타겟으로 사용
    y = df_proc.iloc[:, -1].values
    X = df_proc.iloc[:, :-1].values
    feature_names = df_proc.columns[:-1].tolist()

print(f'\nX 형상: {X.shape}')
print(f'y 형상: {y.shape}')
print(f'피처 개수: {len(feature_names)}')

In [ ]:
# FT-Transformer 모델 정의
class FT_Transformer(nn.Module):
    def __init__(self, input_dim, output_dim=1, d_model=64, n_heads=8,
                 n_layers=4, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.d_model = d_model
        
        # Feature tokenization
        self.feature_tokenizer = nn.Linear(1, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.positional_encoding = nn.Parameter(torch.randn(1, input_dim + 1, d_model))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='relu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # Output head
        self.output_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, output_dim)
        )
    
    def forward(self, x):
        batch_size = x.size(0)
        
        # Tokenize features
        x = x.unsqueeze(-1)  # [batch, features, 1]
        tokens = self.feature_tokenizer(x)  # [batch, features, d_model]
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls_tokens, tokens], dim=1)  # [batch, features+1, d_model]
        
        # Add positional encoding
        tokens = tokens + self.positional_encoding
        
        # Transformer encoding
        transformer_out = self.transformer_encoder(tokens)
        
        # Use CLS token for prediction
        cls_out = transformer_out[:, 0, :]
        output = self.output_head(cls_out)
        
        return output

print('FT-Transformer 클래스 정의 완료')

In [ ]:
# FT-Transformer 학습 함수
def train_ft_transformer(X_train, y_train, X_val, y_val, params, epochs=100, patience=15, verbose=False):
    """FT-Transformer 학습 함수"""
    model = FT_Transformer(
        input_dim=X_train.shape[1],
        d_model=params['d_model'],
        n_heads=params['n_heads'],
        n_layers=params['n_layers'],
        dim_feedforward=params['dim_feedforward'],
        dropout=params['dropout']
    ).to(device)
    
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=params['learning_rate'],
        weight_decay=params['weight_decay']
    )
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=False)
    criterion = nn.MSELoss()
    
    # 데이터 로더 준비
    X_train_tensor = torch.FloatTensor(X_train).to(device)
    y_train_tensor = torch.FloatTensor(y_train).unsqueeze(-1).to(device)
    X_val_tensor = torch.FloatTensor(X_val).to(device)
    y_val_tensor = torch.FloatTensor(y_val).unsqueeze(-1).to(device)
    
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        with torch.no_grad():
            y_pred_val = model(X_val_tensor)
            val_loss = criterion(y_pred_val, y_val_tensor).item()
        
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            if verbose:
                print(f'Early stopping at epoch {epoch+1}')
            break
        
        if verbose and (epoch + 1) % 20 == 0:
            print(f'Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}')
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model

print('FT-Transformer 학습 함수 정의 완료')

In [ ]:
# 예측 함수
def predict_ft_transformer(model, X):
    """FT-Transformer 예측 함수"""
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X).to(device)
        y_pred = model(X_tensor).cpu().numpy().flatten()
    return y_pred

print('예측 함수 정의 완료')

## 1. Base 모델 평가 (StandardScaler 적용)
기본 하이퍼파라미터를 사용하여 모델을 학습하고 평가합니다.

In [ ]:
# Train/Test 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# StandardScaler 적용
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'학습 데이터: {X_train_scaled.shape}')
print(f'테스트 데이터: {X_test_scaled.shape}')

# Base FT-Transformer 파라미터
base_ft_params = {
    'd_model': 64,
    'n_heads': 8,
    'n_layers': 3,
    'dim_feedforward': 256,
    'dropout': 0.1,
    'batch_size': 64,
    'learning_rate': 0.001,
    'weight_decay': 1e-5
}

print('\n=== Base FT-Transformer 학습 시작 ===')
base_ft_model = train_ft_transformer(
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    params=base_ft_params,
    epochs=100,
    patience=15,
    verbose=True
)

# 예측
y_pred_ft_base = predict_ft_transformer(base_ft_model, X_test_scaled)

# 평가
base_ft_mae = mean_absolute_error(y_test, y_pred_ft_base)
base_ft_rmse = np.sqrt(mean_squared_error(y_test, y_pred_ft_base))

print(f'\n=== Base FT-Transformer 결과 ===')
print(f'Base MAE: {base_ft_mae:.4f}')
print(f'Base RMSE: {base_ft_rmse:.4f}')

In [ ]:
# Base TabPFN
print('\n=== Base TabPFN 학습 시작 ===')

# TabPFN은 최대 1000개 샘플과 100개 피처를 처리할 수 있습니다
# 데이터가 크면 샘플링이 필요할 수 있습니다
max_samples = min(1000, X_train_scaled.shape[0])
if X_train_scaled.shape[0] > max_samples:
    print(f'TabPFN 제약으로 인해 {max_samples}개 샘플만 사용합니다.')
    indices = np.random.choice(X_train_scaled.shape[0], max_samples, replace=False)
    X_train_tabpfn = X_train_scaled[indices]
    y_train_tabpfn = y_train[indices]
else:
    X_train_tabpfn = X_train_scaled
    y_train_tabpfn = y_train

base_tabpfn_model = TabPFNRegressor(device=device, N_ensemble_configurations=4)
base_tabpfn_model.fit(X_train_tabpfn, y_train_tabpfn)

# 예측
y_pred_tabpfn_base = base_tabpfn_model.predict(X_test_scaled)

# 평가
base_tabpfn_mae = mean_absolute_error(y_test, y_pred_tabpfn_base)
base_tabpfn_rmse = np.sqrt(mean_squared_error(y_test, y_pred_tabpfn_base))

print(f'\n=== Base TabPFN 결과 ===')
print(f'Base MAE: {base_tabpfn_mae:.4f}')
print(f'Base RMSE: {base_tabpfn_rmse:.4f}')

## 2. Tuned 모델 (5-Fold CV + Optuna)
Optuna를 사용하여 5-fold 교차검증으로 하이퍼파라미터를 최적화합니다.

In [ ]:
# FT-Transformer Optuna 튜닝
def objective_ft_transformer(trial):
    """FT-Transformer 하이퍼파라미터 최적화 목적 함수"""
    params = {
        'd_model': trial.suggest_categorical('d_model', [32, 64, 128]),
        'n_heads': trial.suggest_categorical('n_heads', [4, 8]),
        'n_layers': trial.suggest_int('n_layers', 2, 4),
        'dim_feedforward': trial.suggest_categorical('dim_feedforward', [128, 256, 512]),
        'dropout': trial.suggest_float('dropout', 0.0, 0.3),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-2),
        'weight_decay': trial.suggest_loguniform('weight_decay', 1e-6, 1e-3)
    }
    
    # 5-Fold Cross-Validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled)):
        X_fold_train = X_train_scaled[train_idx]
        y_fold_train = y_train[train_idx]
        X_fold_val = X_train_scaled[val_idx]
        y_fold_val = y_train[val_idx]
        
        # 모델 학습
        model = train_ft_transformer(
            X_fold_train, y_fold_train,
            X_fold_val, y_fold_val,
            params=params,
            epochs=50,  # Optuna에서는 더 빠른 학습을 위해 epochs 줄임
            patience=10,
            verbose=False
        )
        
        # 검증 성능 평가
        y_pred = predict_ft_transformer(model, X_fold_val)
        mae = mean_absolute_error(y_fold_val, y_pred)
        cv_scores.append(mae)
    
    return np.mean(cv_scores)

print('=== FT-Transformer Optuna 튜닝 시작 ===')
study_ft = optuna.create_study(direction='minimize', study_name='ft_transformer')
study_ft.optimize(objective_ft_transformer, n_trials=20, show_progress_bar=True)

print(f'\nBest trial:')
print(f'  Value (MAE): {study_ft.best_trial.value:.4f}')
print(f'  Params:')
for key, value in study_ft.best_trial.params.items():
    print(f'    {key}: {value}')

In [ ]:
# 최적 파라미터로 전체 학습 데이터에 대해 학습
print('\n=== 최적 파라미터로 FT-Transformer 재학습 ===')
best_ft_params = study_ft.best_trial.params

tuned_ft_model = train_ft_transformer(
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    params=best_ft_params,
    epochs=100,
    patience=15,
    verbose=True
)

# 예측 및 평가
y_pred_ft_tuned = predict_ft_transformer(tuned_ft_model, X_test_scaled)
tuned_ft_mae = mean_absolute_error(y_test, y_pred_ft_tuned)
tuned_ft_rmse = np.sqrt(mean_squared_error(y_test, y_pred_ft_tuned))

print(f'\n=== Tuned FT-Transformer 결과 ===')
print(f'Tuned MAE: {tuned_ft_mae:.4f}')
print(f'Tuned RMSE: {tuned_ft_rmse:.4f}')

In [ ]:
# TabPFN Optuna 튜닝
# 참고: TabPFN은 사전학습된 모델이므로 튜닝할 수 있는 하이퍼파라미터가 제한적입니다.
# 주로 앙상블 구성 수를 조정할 수 있습니다.

def objective_tabpfn(trial):
    """TabPFN 하이퍼파라미터 최적화 목적 함수"""
    n_ensemble = trial.suggest_int('N_ensemble_configurations', 2, 8)
    
    # 5-Fold Cross-Validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled)):
        X_fold_train = X_train_scaled[train_idx]
        y_fold_train = y_train[train_idx]
        X_fold_val = X_train_scaled[val_idx]
        y_fold_val = y_train[val_idx]
        
        # TabPFN 제약 적용
        max_samples = min(1000, X_fold_train.shape[0])
        if X_fold_train.shape[0] > max_samples:
            indices = np.random.choice(X_fold_train.shape[0], max_samples, replace=False)
            X_fold_train = X_fold_train[indices]
            y_fold_train = y_fold_train[indices]
        
        # 모델 학습
        model = TabPFNRegressor(device=device, N_ensemble_configurations=n_ensemble)
        model.fit(X_fold_train, y_fold_train)
        
        # 검증 성능 평가
        y_pred = model.predict(X_fold_val)
        mae = mean_absolute_error(y_fold_val, y_pred)
        cv_scores.append(mae)
    
    return np.mean(cv_scores)

print('\n=== TabPFN Optuna 튜닝 시작 ===')
study_tabpfn = optuna.create_study(direction='minimize', study_name='tabpfn')
study_tabpfn.optimize(objective_tabpfn, n_trials=5, show_progress_bar=True)

print(f'\nBest trial:')
print(f'  Value (MAE): {study_tabpfn.best_trial.value:.4f}')
print(f'  Params:')
for key, value in study_tabpfn.best_trial.params.items():
    print(f'    {key}: {value}')

In [ ]:
# 최적 파라미터로 전체 학습 데이터에 대해 학습
print('\n=== 최적 파라미터로 TabPFN 재학습 ===')
best_n_ensemble = study_tabpfn.best_trial.params['N_ensemble_configurations']

# TabPFN 제약 적용
max_samples = min(1000, X_train_scaled.shape[0])
if X_train_scaled.shape[0] > max_samples:
    print(f'TabPFN 제약으로 인해 {max_samples}개 샘플만 사용합니다.')
    indices = np.random.choice(X_train_scaled.shape[0], max_samples, replace=False)
    X_train_tabpfn = X_train_scaled[indices]
    y_train_tabpfn = y_train[indices]
else:
    X_train_tabpfn = X_train_scaled
    y_train_tabpfn = y_train

tuned_tabpfn_model = TabPFNRegressor(device=device, N_ensemble_configurations=best_n_ensemble)
tuned_tabpfn_model.fit(X_train_tabpfn, y_train_tabpfn)

# 예측 및 평가
y_pred_tabpfn_tuned = tuned_tabpfn_model.predict(X_test_scaled)
tuned_tabpfn_mae = mean_absolute_error(y_test, y_pred_tabpfn_tuned)
tuned_tabpfn_rmse = np.sqrt(mean_squared_error(y_test, y_pred_tabpfn_tuned))

print(f'\n=== Tuned TabPFN 결과 ===')
print(f'Tuned MAE: {tuned_tabpfn_mae:.4f}')
print(f'Tuned RMSE: {tuned_tabpfn_rmse:.4f}')

## 3. 결과 비교 테이블
PPT 이미지와 같은 형식으로 결과를 정리합니다.

In [ ]:
# 최종 결과 테이블 생성
results_table = pd.DataFrame({
    'Models': ['FT-Transformer', 'TabPFN'],
    'Base MAE': [base_ft_mae, base_tabpfn_mae],
    'Base RMSE': [base_ft_rmse, base_tabpfn_rmse],
    'Tuned MAE': [tuned_ft_mae, tuned_tabpfn_mae],
    'Tuned RMSE': [tuned_ft_rmse, tuned_tabpfn_rmse],
    'Setup': ['5-fold / Optuna / StandardScaler', '5-fold / Optuna / StandardScaler']
})

print('\n' + '='*100)
print('모델 성능 비교 결과')
print('='*100)
print(results_table.to_string(index=False))
print('='*100)

# 결과를 CSV 파일로 저장
results_table.to_csv('model_comparison_results.csv', index=False)
print('\n결과가 model_comparison_results.csv 파일로 저장되었습니다.')

In [ ]:
# 최고 성능 모델 강조
print('\n' + '='*100)
print('최고 성능 모델 (Base & Tuned)')
print('='*100)

# Base 모델에서 최고 성능
if base_ft_mae < base_tabpfn_mae:
    print(f'✅ Base 최고 MAE: FT-Transformer ({base_ft_mae:.4f})')
else:
    print(f'✅ Base 최고 MAE: TabPFN ({base_tabpfn_mae:.4f})')

if base_ft_rmse < base_tabpfn_rmse:
    print(f'✅ Base 최고 RMSE: FT-Transformer ({base_ft_rmse:.4f})')
else:
    print(f'✅ Base 최고 RMSE: TabPFN ({base_tabpfn_rmse:.4f})')

print()

# Tuned 모델에서 최고 성능
if tuned_ft_mae < tuned_tabpfn_mae:
    print(f'✅ Tuned 최고 MAE: FT-Transformer ({tuned_ft_mae:.4f})')
else:
    print(f'✅ Tuned 최고 MAE: TabPFN ({tuned_tabpfn_mae:.4f})')

if tuned_ft_rmse < tuned_tabpfn_rmse:
    print(f'✅ Tuned 최고 RMSE: FT-Transformer ({tuned_ft_rmse:.4f})')
else:
    print(f'✅ Tuned 최고 RMSE: TabPFN ({tuned_tabpfn_rmse:.4f})')

print('='*100)

# 개선도 계산
print('\n' + '='*100)
print('Tuning으로 인한 개선도')
print('='*100)

ft_mae_improvement = ((base_ft_mae - tuned_ft_mae) / base_ft_mae) * 100
ft_rmse_improvement = ((base_ft_rmse - tuned_ft_rmse) / base_ft_rmse) * 100
tabpfn_mae_improvement = ((base_tabpfn_mae - tuned_tabpfn_mae) / base_tabpfn_mae) * 100
tabpfn_rmse_improvement = ((base_tabpfn_rmse - tuned_tabpfn_rmse) / base_tabpfn_rmse) * 100

print(f'FT-Transformer MAE 개선: {ft_mae_improvement:+.2f}%')
print(f'FT-Transformer RMSE 개선: {ft_rmse_improvement:+.2f}%')
print(f'TabPFN MAE 개선: {tabpfn_mae_improvement:+.2f}%')
print(f'TabPFN RMSE 개선: {tabpfn_rmse_improvement:+.2f}%')
print('='*100)

## 4. 요약

### 실행된 작업
1. **데이터 전처리**: icu_final_feature_set.csv 로드 및 StandardScaler 적용
2. **Base 모델 학습**: 기본 하이퍼파라미터로 FT-Transformer와 TabPFN 학습
3. **하이퍼파라미터 튜닝**: Optuna를 사용한 5-fold 교차검증
4. **Tuned 모델 학습**: 최적 파라미터로 재학습
5. **성능 비교**: Base MAE/RMSE와 Tuned MAE/RMSE 비교

### 핵심 포인트
- **FT-Transformer**: Feature Tokenization + Transformer 아키텍처
- **TabPFN**: 사전학습된 메타러닝 기반 모델 (최대 1000 샘플, 100 피처 제약)
- **5-Fold CV**: 교차검증으로 안정적인 성능 평가
- **Optuna**: 자동 하이퍼파라미터 최적화
- **StandardScaler**: 특성 정규화로 모델 성능 향상

### 주의사항
- TabPFN은 데이터 크기 제약이 있어 샘플링이 필요할 수 있습니다
- FT-Transformer는 GPU 사용 시 학습 속도가 크게 향상됩니다
- Optuna 튜닝 시 n_trials를 늘리면 더 나은 성능을 얻을 수 있습니다